In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, trim, row_number, current_date, current_timestamp, sha2, concat_ws
from pyspark.sql.window import Window
import urllib

# ==========================================
# 1. PARAMETERIZATION & CONFIGURATION
# ==========================================
# Senior engineers NEVER hardcode paths or variables. We use widgets.
dbutils.widgets.text("company_key", "ABC", "Company Key")
dbutils.widgets.text("brand_key", "ABC", "Brand Key")
dbutils.widgets.text("catalog_name", "erp_lakehouse", "Catalog Name")
dbutils.widgets.text("schema_name", "silver", "Schema Name")
dbutils.widgets.text("silver_external_path", "abfss://silver@bbmanufacturingprod.dfs.core.windows.net/delta/sl_productline", "Silver External Path")
dbutils.widgets.text("exception_external_path", "abfss://silver@bbmanufacturingprod.dfs.core.windows.net/delta/exception_table", "Exception Table Path")


COMPANY_KEY = dbutils.widgets.get("company_key")
BRAND_KEY = dbutils.widgets.get("brand_key")
CATALOG_NAME= dbutils.widgets.get("catalog_name")
SILVER_SCHEMA= dbutils.widgets.get("schema_name")
SILVER_PATH = dbutils.widgets.get("silver_external_path")
EXCEPTION_PATH = dbutils.widgets.get("exception_external_path")



In [0]:
# ==========================================
# 2. DATA LOADING (BRONZE LAYER)
# ==========================================
print("Reading data from Bronze Delta Table...")
bronze_df = spark.read.table(f"{CATALOG_NAME}.bronze.bz_itemcategories")

# Inject control columns early
enriched_df = bronze_df \
    .withColumn('RecordStatus', lit('0')) \
    .withColumn('CompanyKey', lit(COMPANY_KEY)) \
    .withColumn('BrandKey', lit(BRAND_KEY))


In [0]:
# ==========================================
# 3. DATA QUALITY & INTEGRITY RULES (SILVER AUDIT)
# ==========================================
# Rule 1: Identify Null or Blank IDs (Highest Priority Exception)
validated_df = enriched_df.withColumn(
    "RecordStatus",
    when((col("id").isNull()) | (trim(col("id")) == ""), lit('2')).otherwise(col("RecordStatus"))
)

# Rule 2: Identify Duplicates using Windowing (Only on valid IDs to save performance)
# Note: We order by ingestion_time or LastModifiedDateTime descending if available, to keep the latest record!
window_spec = Window.partitionBy("id", "CompanyKey", "BrandKey").orderBy(col("ingestion_time").desc())

processed_df = validated_df.withColumn("row_num", row_number().over(window_spec)) \
    .withColumn("RecordStatus", when((col("RecordStatus") == '0') & (col("row_num") > 1), lit('1')).otherwise(col("RecordStatus"))) \
    .drop("row_num")

# Cache this dataframe because we are going to split it into two independent forks (Exceptions vs Clean Data)
processed_df.cache()


In [0]:
# ==========================================
# 4. EXCEPTION HANDLING (THE DE ROUTINE)
# ==========================================
print("Processing and routing data exceptions...")

# Create the dedicated silver schema isolation layer
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")

# Ensure exception external table exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}.exception_table (
    ExceptionID STRING,
    BrandKey STRING,
    RecordKey STRING,
    TableName STRING,
    ColumnName STRING,
    ExceptionDetails STRING,
    SysCreatedDate DATE,
    SysCreatedBy STRING
)
USING DELTA
LOCATION '{EXCEPTION_PATH}'
""")

# Route Duplicates (Status 1) & Nulls (Status 2)
duplicate_exceptions = processed_df.filter(col("RecordStatus") == '1') \
    .select(
        sha2(concat_ws("||", col("id"), current_timestamp()), 256).alias("ExceptionID"),
        lit(BRAND_KEY).alias("BrandKey"),
        col("id").cast("string").alias("RecordKey"),
        lit("bz_itemcategories").alias("TableName"),
        lit("").alias("ColumnName"),
        lit("Duplicate record found during deduplication").alias("ExceptionDetails"),
        current_date().alias("SysCreatedDate"),
        lit("databricks_job").alias("SysCreatedBy")
    )

null_exceptions = processed_df.filter(col("RecordStatus") == '2') \
    .select(
        sha2(concat_ws("||", col("id"), current_timestamp()), 256).alias("ExceptionID"),
        lit(BRAND_KEY).alias("BrandKey"),
        lit("UNKNOWN_KEY").alias("RecordKey"),
        lit("bz_itemcategories").alias("TableName"),
        lit("id").alias("ColumnName"),
        lit("Null or Blank primary key 'id' identified").alias("ExceptionDetails"),
        current_date().alias("SysCreatedDate"),
        lit("databricks_job").alias("SysCreatedBy")
    )

# Union and append to external exception logs
final_exceptions_df = duplicate_exceptions.unionByName(null_exceptions)
if final_exceptions_df.count() > 0:
    final_exceptions_df.write.mode("append").saveAsTable("erp_lakehouse.silver.exception_table")


In [0]:
# ==========================================
# 5. TRANSFORM CLEAN DATA FOR SILVER LAYER
# ==========================================

from pyspark.sql.functions import coalesce, col, lit, when


print("Transforming valid corporate records...")
clean_records_df = processed_df.filter(col("RecordStatus") == '0')


print("Transforming corporate Product Line records with null elimination... 🚀")

final_silver_df = clean_records_df.select(
    col("CompanyKey").cast("string"),
    col("BrandKey").cast("string"),
    
    # Primary Key - explicitly casted
    col("id").alias("ProductLnSrcId").cast("string"),
    
    # Applying coalesce for safe string default fallbacks
    coalesce(col("code"), lit("")).alias("ProductLnKey").cast("string"),
    coalesce(col("displayName"), lit("")).alias("ProductLnDesc").cast("string"),
    
    # Hardcoded metadata flags
    lit("P").alias("ProductLnType").cast("string"),
    lit("Parts").alias("ProductLnTypeDesc").cast("string"),
    
    # Safe explicit null handling for missing source creation fields
    lit(None).cast("timestamp").alias("SourceCreatedTime"),
    
    # Operational Audit Logs
    col("LastModifiedDateTime").alias("SourceUpdatedTime").cast("timestamp"),
    col("ingestion_time").alias("SysCreatedTime").cast("timestamp"),
    col("RecordStatus").cast("string")
)

display(final_silver_df)


In [0]:
# ==========================================
# 6. EXTERNAL DELTA LAKE MERGE OPERATION
# ==========================================
print("Executing Idempotent Delta Merge into Silver Layer...")

# Create External Silver Table with accurate explicit mapping schema
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}.sl_productline (
    CompanyKey STRING,
    BrandKey STRING,
    ProductLnSrcId STRING,
    ProductLnKey STRING,
    ProductLnDesc STRING,
    ProductLnType STRING,
    ProductLnTypeDesc STRING,
    SourceCreatedTime TIMESTAMP,
    SourceUpdatedTime TIMESTAMP,
    SysCreatedTime TIMESTAMP,
    RecordStatus STRING
)
USING DELTA
LOCATION '{SILVER_PATH}'
""")

In [0]:
# Register temporary view for the merge execution
final_silver_df.createOrReplaceTempView("final_silver_src")

spark.sql("""
MERGE INTO erp_lakehouse.silver.sl_productline AS tgt
USING final_silver_src AS src
ON tgt.ProductLnSrcId = src.ProductLnSrcId 
   AND tgt.CompanyKey = src.CompanyKey 
   AND tgt.BrandKey = src.BrandKey
WHEN MATCHED THEN
  UPDATE SET
    tgt.ProductLnKey = src.ProductLnKey,
    tgt.ProductLnDesc = src.ProductLnDesc,
    tgt.SourceUpdatedTime = src.SourceUpdatedTime
WHEN NOT MATCHED THEN
  INSERT (
    CompanyKey, BrandKey, ProductLnSrcId, ProductLnKey, ProductLnDesc,
    ProductLnType, ProductLnTypeDesc, SourceCreatedTime, SourceUpdatedTime,
    SysCreatedTime, RecordStatus
  )
  VALUES (
    src.CompanyKey, src.BrandKey, src.ProductLnSrcId, src.ProductLnKey, src.ProductLnDesc,
    src.ProductLnType, src.ProductLnTypeDesc, src.SourceCreatedTime, src.SourceUpdatedTime,
    src.SysCreatedTime, src.RecordStatus
  )
""")

# Uncache data to free memory cluster resources
processed_df.unpersist()
print("Pipeline executed successfully and cleanly.")